# 🔢 Topological Sort — Runnable Notebook

Companion to [`README.md`](README.md) and
[`10_topological_sort.html`](10_topological_sort.html).

Order a **DAG** so every edge points forward — via **Kahn's (BFS)** and **DFS**.

## 1. Kahn's algorithm — peel off in-degree-0 tasks

In [ ]:
from collections import deque

def topo_sort_kahn(n, edges):
    """n vertices 0..n-1; edges = (u, v) meaning u must come before v.
       Returns a valid order, or None if there is a cycle."""
    indeg = [0] * n
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1                      # v gains a prerequisite

    q = deque(i for i in range(n) if indeg[i] == 0)   # everything ready now
    order = []
    while q:
        u = q.popleft()
        order.append(u)                    # no prerequisites left -> output it
        for v in adj[u]:
            indeg[v] -= 1                  # u done -> v loses a prerequisite
            if indeg[v] == 0:
                q.append(v)                # v just became ready
    return order if len(order) == n else None   # short -> a cycle blocked some tasks

edges = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]
order = topo_sort_kahn(5, edges)
print("Kahn order:", order)

## 2. Verify an order is valid
Every edge `u→v` must have `u` appear **before** `v`.

In [ ]:
def is_valid_topo(order, edges):
    """True if every edge points forward in the given order."""
    if order is None:
        return False
    pos = {v: i for i, v in enumerate(order)}   # vertex -> its position
    return all(pos[u] < pos[v] for u, v in edges)

print("Kahn order valid?", is_valid_topo(order, edges))
assert is_valid_topo(order, edges)

## 3. DFS-based — push on finish, then reverse
Also reports cycles via 3 states (unseen / in-progress / done).

In [ ]:
def topo_sort_dfs(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    visited = [0] * n                      # 0 unseen, 1 in-progress, 2 done
    order, ok = [], True
    def dfs(u):
        nonlocal ok
        visited[u] = 1                     # on the recursion stack
        for v in adj[u]:
            if visited[v] == 1:            # edge back to an in-progress node -> cycle
                ok = False
            elif visited[v] == 0:
                dfs(v)
        visited[u] = 2                     # finished
        order.append(u)                    # push on finish
    for i in range(n):
        if visited[i] == 0:
            dfs(i)
    return order[::-1] if ok else None     # reverse of finish order

order_dfs = topo_sort_dfs(5, edges)
print("DFS order:", order_dfs)
assert is_valid_topo(order_dfs, edges)     # a (possibly different) valid order

## 4. A cycle has no topological order

In [ ]:
cyclic = [(0, 1), (1, 2), (2, 0)]          # 0 -> 1 -> 2 -> 0
print("Kahn on a cycle:", topo_sort_kahn(3, cyclic))
print("DFS  on a cycle:", topo_sort_dfs(3, cyclic))
assert topo_sort_kahn(3, cyclic) is None
assert topo_sort_dfs(3, cyclic) is None

## ✅ Recap
- Works on a **DAG** only; edge `u→v` = **u before v**.
- **Kahn's**: output in-degree-0 tasks, decrement dependents; output `< V` ⇒ **cycle**.
- **DFS**: push on **finish**, then **reverse**; edge to an **in-progress** node ⇒ **cycle**.
- Usually **many** valid orders; both run in `O(V + E)`.

Next: [`11_Cycle_Detection`](../11_Cycle_Detection/README.md).